# Solution A2 · Gradient boosting for classification: gradient, hessian and Newton step

**Type**: Implement from scratch

## Objective

Implement boosting with the logistic loss: initial prediction as log-odds, **gradient** ($y - p$) as the target of the tree and leaf values with the **Newton step** ($\sum r / \sum p(1-p)$, that is gradient divided by hessian). Check that it matches `GradientBoostingClassifier` in the first rounds and measure how much
a plain gradient step (without the Newton factor) departs.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = make_classification(1500, n_features=8, n_informative=4, class_sep=0.8, flip_y=0.05, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

## Commented solution

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def fit_gbc(X, y, n_rounds, eta, depth, newton=True):
    p0 = y.mean()
    F0 = np.log(p0 / (1 - p0))                                              # log-odds of the prevalence
    F = np.full(len(y), F0)
    trees = []
    for _ in range(n_rounds):
        p = sigmoid(F)
        r = y - p                                                            # negative gradient of the log-loss
        tree = DecisionTreeRegressor(max_depth=depth, criterion="squared_error", random_state=0).fit(X, r)
        leaves = tree.apply(X)
        values = {}
        for l in np.unique(leaves):
            m = leaves == l
            hessian = (p[m] * (1 - p[m])).sum()
            values[l] = r[m].sum() / max(hessian, 1e-12) if newton else r[m].mean()      # Newton step: gradient / hessian
        F = F + eta * np.array([values[l] for l in leaves])
        trees.append((tree, values))
    return F0, trees


def predict_proba(model, X, eta):
    F0, trees = model
    F = np.full(len(X), F0)
    for tree, values in trees:
        F = F + eta * np.array([values[l] for l in tree.apply(X)])
    return sigmoid(F)

In [ ]:
def attempt(B):
    mine = predict_proba(fit_gbc(X_tr, y_tr, B, 0.1, 3), X_te, 0.1)
    sk = GradientBoostingClassifier(n_estimators=B, learning_rate=0.1, max_depth=3, criterion="squared_error", random_state=0).fit(X_tr, y_tr)
    p_sk = sk.predict_proba(X_te)[:, 1]
    return mine, p_sk


m3, s3 = attempt(3)
m50, s50 = attempt(50)
diff_3 = float(np.abs(m3 - s3).max())
diff_50 = float(np.abs(m50 - s50).max())
mean_diff_50 = float(np.abs(m50 - s50).mean())
class_agreement_50 = float(((m50 > 0.5) == (s50 > 0.5)).mean())
print(f"maximum difference: 3 rounds {diff_3:.1e}, 50 rounds {diff_50:.3f} | mean after 50 rounds {mean_diff_50:.5f} | same classes: {class_agreement_50:.1%}")

In [ ]:
p_newton = predict_proba(fit_gbc(X_tr, y_tr, 50, 0.1, 3, newton=True), X_te, 0.1)
p_grad = predict_proba(fit_gbc(X_tr, y_tr, 50, 0.1, 3, newton=False), X_te, 0.1)
acc_with_newton, acc_without_newton = float(((p_newton > 0.5) == y_te).mean()), float(((p_grad > 0.5) == y_te).mean())
logloss_with_newton, logloss_without_newton = log_loss(y_te, p_newton), log_loss(y_te, p_grad)
print(f"with the Newton step: accuracy {acc_with_newton:.3f}, log-loss {logloss_with_newton:.3f} | gradient only: accuracy {acc_without_newton:.3f}, log-loss {logloss_without_newton:.3f}")

## Check

In [ ]:
with verify():
    check(close(sigmoid(np.array([0.0, 30.0, -30.0])), [0.5, 1.0, 0.0], atol=1e-9), "correct sigmoid", "1 / (1 + exp(-z))")
    m = fit_gbc(X_tr, y_tr, 1, 0.1, 3)
    check(len(m[1]) == 1 and close(m[0], np.log(y_tr.mean() / (1 - y_tr.mean()))), "a single round, F0 = log-odds of the prevalence", "F0 = log(p0 / (1 - p0))")
    check(diff_3 < 1e-9, "after 3 rounds the probabilities match scikit-learn (1e-9)", "leaf values: sum of the residuals / sum of p (1 - p)")
    check(mean_diff_50 < 0.005 and class_agreement_50 > 0.99, "after 50 rounds the mean difference is tiny (below 0.005) and the classes match for over 99%", "differences from float32 and tie-breaking")
    check(acc_without_newton > 0.7 and abs(acc_with_newton - acc_without_newton) < 0.05 and logloss_without_newton > logloss_with_newton + 0.05, "without the Newton factor the log-loss is worse by over 0.05 (the steps are too small), the accuracy almost the same", "compare the two versions")
done()

## What you learned

- **Logistic boosting uses the gradient to choose the structure and the hessian for the leaf value.** The tree is trained on the residuals $y - p$; the value of a leaf is $\sum r/\sum p(1-p)$: the Newton step to minimize the log-loss in that leaf. It matches scikit-learn in the first rounds to 1e-9; after 50 rounds the differences are on average 0.0006 (at most 0.14 on a single example) and the predicted classes match 100%.
- **The hessian $p(1-p)$ is small for "safe" examples** (p close to 0 or 1) and large for uncertain ones: the leaf "trusts" the examples near the boundary more. It is the foundation of `min_child_weight` in XGBoost: a minimum for the **sum of the hessians** in a leaf, not for the number of examples.
- **Without the Newton factor** (value = mean of the residuals) the steps are too small (the residuals $y - p$ are in [−1, 1], but the right step on the log-odds scale is much larger): after the same number of rounds the log-loss is much worse (0.48 against 0.37) and the accuracy drops too (0.80 against 0.84). It is the lesson: the class reads fine even with an undertrained model, the **probabilities** do not.
- XGBoost extends the idea: it uses a second-order expansion of the loss **plus a penalty on the leaf weights** ($\lambda$): the optimal leaf value becomes $-\sum g/(\sum h + \lambda)$ and the gain of a split is a formula in $G$ and $H$.

## Common pitfall

Interpreting the raw predictions of a boosting model as probabilities without the sigmoid, or using a "simplified" implementation (value = mean of the residuals) and looking only at the accuracy: the probabilities come out squashed toward the prevalence.

## Interview question

<details><summary>Why does XGBoost also use the hessian?</summary>

Because it approximates the loss with a second-order expansion: the optimal value of a leaf is -G/(H+λ), where G is the sum of the gradients and H that of the hessians; it is the Newton step, much more efficient than a plain gradient step, and it lets you compute the gain of every split in closed form.
The hessian also gives the meaning of min_child_weight: a minimum weight per leaf, where very confident examples count less.

</details>